# SHAP Values — Shapley Additive Explanations

> Lundberg, S. & Lee, S.-I. (2017) *"A Unified Approach to Interpreting Model Predictions"*

---

## Why SHAP?

Most ML models are **black boxes** — they predict well but don't explain *why*.  
SHAP provides **consistent, local, and global** explanations using a game-theory foundation.

---

## Shapley Value — Game Theory Foundation

In cooperative game theory, the **Shapley value** is the fair share of a reward to assign each player, averaged over all possible orderings in which players join the coalition:

$$\phi_i(v) = \sum_{S \subseteq F \setminus \{i\}} \frac{|S|!(|F|-|S|-1)!}{|F|!} \left[v(S \cup \{i\}) - v(S)\right]$$

where:
- $F$ = set of all features
- $S$ = coalition of features without feature $i$
- $v(S)$ = model prediction using only features in $S$
- $\phi_i$ = SHAP value for feature $i$

### SHAP Explanation Model

$$f(x) \approx g(x') = \phi_0 + \sum_{i=1}^M \phi_i x'_i$$

where $\phi_0 = E[f(X)]$ is the global baseline and each $\phi_i$ is how much feature $i$ pushed the prediction from baseline.

### Key properties

| Property | Meaning |
|---|---|
| **Efficiency** | $\sum_i \phi_i = f(x) - E[f(X)]$ |
| **Symmetry** | Equal contribution → equal SHAP value |
| **Dummy** | Feature with zero contribution → SHAP = 0 |
| **Additivity** | SHAP of an ensemble = sum of SHAPs of base models |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import shap
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

shap.initjs()
np.random.seed(42)
print('Imports OK')

---
## 1  Dataset — Breast Cancer (Wisconsin)

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target  # 0=malignant, 1=benign

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'Target: malignant={sum(y==0)}, benign={sum(y==1)}')

---
## 2  Train a Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print(f'Test accuracy: {rf.score(X_test, y_test):.4f}')

---
## 3  TreeExplainer — Exact SHAP for Tree Models

In [ ]:
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

# shap_values is a list [class_0_shap, class_1_shap]
# Use class 1 (benign)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

print(f'SHAP values shape: {sv.shape}')  # (n_test, n_features)
print(f'Base value (E[f(X)]): {explainer.expected_value[1]:.4f}')

# Efficiency check: SHAP sum + base ≈ model output
pred_proba = rf.predict_proba(X_test)[:, 1]
shap_sum = sv.sum(axis=1) + explainer.expected_value[1]
print(f'Max reconstruction error: {np.abs(shap_sum - pred_proba).max():.6f}')

---
## 4  Global Explanations

In [ ]:
# --- Bar plot: mean |SHAP| per feature ---
shap.summary_plot(sv, X_test, plot_type='bar', max_display=15,
                  show=False)
plt.title('Global Feature Importance (mean |SHAP|)')
plt.tight_layout()
plt.show()

In [ ]:
# --- Beeswarm plot: distribution of SHAP values ---
shap.summary_plot(sv, X_test, max_display=15, show=False)
plt.title('SHAP Beeswarm — Direction and Magnitude')
plt.tight_layout()
plt.show()

---
## 5  Local Explanations — Single Prediction

In [ ]:
# Pick one test sample and explain it
sample_idx = 0
sample = X_test.iloc[[sample_idx]]
pred = rf.predict(sample)[0]
proba = rf.predict_proba(sample)[0, 1]

print(f'True label : {y_test[sample_idx]}  ({data.target_names[y_test[sample_idx]]})')
print(f'Predicted  : {pred}  ({data.target_names[pred]})')
print(f'P(benign)  : {proba:.4f}')

# Waterfall plot
shap_exp = shap.Explanation(
    values=sv[sample_idx],
    base_values=explainer.expected_value[1],
    data=sample.values[0],
    feature_names=list(X_test.columns)
)
shap.waterfall_plot(shap_exp, max_display=12, show=False)
plt.title(f'SHAP Waterfall — Sample {sample_idx}')
plt.tight_layout()
plt.show()

---
## 6  Dependence Plot — Feature Interaction

In [ ]:
# Top feature from global importance
top_feature = X_test.columns[np.argsort(np.abs(sv).mean(axis=0))[-1]]
print(f'Top feature: {top_feature}')

shap.dependence_plot(top_feature, sv, X_test, show=False)
plt.title(f'SHAP Dependence Plot — {top_feature}')
plt.tight_layout()
plt.show()

---
## 7  SHAP for Linear Model — LinearExplainer

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

lr = LogisticRegression(max_iter=2000, random_state=42)
lr.fit(X_train_s, y_train)
print(f'Logistic Regression test acc: {lr.score(X_test_s, y_test):.4f}')

linear_explainer = shap.LinearExplainer(lr, X_train_s)
sv_lr = linear_explainer.shap_values(X_test_s)

# For binary classification LinearExplainer returns 1D array (log-odds)
if sv_lr.ndim == 1:
    sv_lr = sv_lr.reshape(1, -1).T

shap.summary_plot(sv_lr, pd.DataFrame(X_test_s, columns=X_test.columns),
                  plot_type='bar', max_display=12, show=False)
plt.title('Linear Model — SHAP Feature Importance')
plt.tight_layout()
plt.show()

---
## 8  SHAP vs Permutation Importance — Comparison

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_test, y_test, n_repeats=20, random_state=42)
perm_imp = perm.importances_mean
shap_imp  = np.abs(sv).mean(axis=0)

top_k = 10
top_idx_shap = np.argsort(shap_imp)[-top_k:][::-1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
feat_names = [X_test.columns[i] for i in top_idx_shap]

axes[0].barh(feat_names[::-1], shap_imp[top_idx_shap][::-1], color='steelblue')
axes[0].set_title('SHAP Importance (mean |ϕ|)')

perm_top = perm_imp[top_idx_shap]
axes[1].barh(feat_names[::-1], perm_top[::-1], color='darkorange')
axes[1].set_title('Permutation Importance')

plt.tight_layout()
plt.show()

---
## Summary

| Plot | What it shows |
|---|---|
| **Bar** | Global mean \|SHAP\| per feature |
| **Beeswarm** | Distribution + direction of impact per feature |
| **Waterfall** | Single prediction breakdown from baseline |
| **Dependence** | How one feature's SHAP changes with its value |

SHAP satisfies the three desirable axioms of feature attribution: **local accuracy, missingness, consistency** — unlike permutation importance or Gini importance.

**See also:** Topic 04 (random forest), Topic 24 (SVM), Topic 115 (real-world project)